# Lab 04 · Tool calling: an agent with three tools

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` and save its key in Colab Secrets. DeepSeek uses `DEEPSEEK_API_KEY`; Gemini and FreeInference use their respective secrets. `client` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import Client, PROVIDERS, secret
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
settings = PROVIDERS[PROVIDER]
client = Client(model=settings["model"], family=settings["family"],
                base_url=settings["base_url"], api_key=secret(settings["secret_name"]),
                max_calls=100, max_tokens=256, extra_body=settings.get("extra_body"))
ask = client.ask

## Warm-up · Give the model a small tool

### Warm-up step 1 · Ask without tools

Ask the model for the current time, year, and weather in Rome. This call gives it no clock or weather lookup. It may guess, say it does not know, or give an apparently correct answer; a plausible answer alone does not prove it has live data. Compare it with a trusted clock and weather source.

In [ ]:
prompt = "What is the current time, year, and weather in Rome, Italy?"
answer = ask(prompt)
print("Question:", prompt)
print("Model answer without tools:", answer)

### Warm-up step 2 · Give the model a weather tool

Python can fetch current weather from Open-Meteo. The model requests `get_weather`; our code runs it and sends the actual result back. This is a simple, manual tool-calling example, not an automatic SDK tool call. The lookup needs internet access. Change the coordinates and city together to try another location. The result includes the observation time, temperature in °C, and a numeric weather code. The model should not guess what an unfamiliar code means.

In [ ]:
from urllib.request import urlopen
import json

def get_weather():
    url = ("https://api.open-meteo.com/v1/forecast?latitude=41.9028"
           "&longitude=12.4964&current=temperature_2m,weather_code"
           "&timezone=Europe%2FRome")
    with urlopen(url, timeout=15) as response:
        return json.load(response)["current"]

choice = ask("Available tool: get_weather (current weather in Rome). "
             "Question: What is the weather in Rome? Reply only get_weather or no_tool.")
print("Model request:", choice)
if choice.strip().lower().strip(' ."') == "get_weather":
    weather = get_weather()
    print("Live tool result:", weather)
    print("Model answer:", ask(f"Current weather in Rome: {weather}. State the temperature in °C and observation time. Report the weather code; do not guess its meaning."))
else:
    print("The model did not request the tool. Run this cell again or inspect its reply.")

**Next:** Now give an agent three small tools for working with microservice information.

## Give an agent three small tools
**Problem:** A model cannot read private data unless the application provides it. Even when data is available, we may need to check it and present it consistently.

**In this lab:** The agent can extract a microservice record, validate it, and format it as HTML. The records are local teaching examples, not live service data. The model chooses actions; Python runs the tools and returns their results.

**At a glance**

![Lab 04 at a glance: Question, then Agent chooses (system lists tools), then Extract (service record), then Validate (required fields), then Format HTML (escaped text), then Answer (from tool results).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/04.svg)

## Think about the example

A microservice owns its data and exposes information through a defined interface. The local service records in this example are illustrative.

**Concepts:** Database per service; Service-owned data; Published interface.

Use the scenario above to consider these questions before running the code:

1. Why should a service expose its data through an interface rather than let other services access its database directly?
2. What does checking that a service record has required fields tell you, and what does it not prove?

## The idea in context

A tool is an ordinary function the agent can request. The application runs it and returns its result.

A microservice owns its data and exposes useful information through an interface.

| Key term | Meaning |
| --- | --- |
| **Tool** | A function the model can ask the application to run. |
| **System prompt** | Instructions that describe the agent and its available tools. |
| **Validation** | Checking whether data satisfies defined rules. |
| **Tool result** | The information returned to the model after a tool runs. |

## Pattern · Tool-using agent

A tool is an ordinary function made available to the agent. Here the model selects a tool, Python executes it, and the model sees the result before choosing its next action. The model cannot execute Python by itself. This is a small manual tool loop; SDKs can also provide native tool calling.

## Main exercise at a glance

1. Create a tool to extract data
2. Create a tool to validate data
3. Create a tool to format HTML
4. Tell the agent about its tools
5. Let the agent choose and use tools

## Step 1 · Create a tool to extract data
The tool reads one record from a tiny local catalog. Try changing the service name; an unknown service returns an empty record.

In [ ]:
services = {
    "orders": {"name": "Orders", "owner": "Sales team", "responsibility": "Manage orders"},
    "inventory": {"name": "Inventory", "owner": "Stock team", "responsibility": "Track stock"},
}

def extract_service(name):
    return services.get(name.lower().strip(), {}).copy()

print("Extracted:", extract_service("orders"))

## Step 2 · Create a tool to validate data
A useful record has a name, an owner, and a responsibility. Validation is a deterministic check; it does not prove the record is factually correct.

In [ ]:
def validate_service(data):
    for field in ("name", "owner", "responsibility"):
        if not isinstance(data.get(field), str) or not data[field].strip():
            return "Invalid: missing " + field
    return "Valid"

print("Complete record:", validate_service(extract_service("orders")))
print("Missing owner:", validate_service({"name": "Orders"}))

## Step 3 · Create a tool to format HTML
This tool formats the extracted record. `escape` treats text as text, so special characters cannot become HTML markup.

In [ ]:
from html import escape

def format_html(data):
    name = escape(data["name"])
    owner = escape(data["owner"])
    responsibility = escape(data["responsibility"])
    return f"<h2>{name}</h2><p>Owner: {owner}</p><p>Responsibility: {responsibility}</p>"

print(format_html(extract_service("orders")))

## Step 4 · Tell the agent about its tools
The system prompt lists the tools, the expected reply, and the order for creating HTML. The user can ask a different question in Step 5. The catalog contains only Orders and Inventory; the agent should not invent records.

In [ ]:
system_prompt = """You are a helpful microservices assistant.
You have these tools:
- extract_service: look up Orders or Inventory in the local catalog.
- validate_service: check the latest extracted record.
- format_html: turn the latest validated record into HTML.

Choose one action at a time. Reply with a JSON object only:
{ "tool": "extract_service", "name": "orders" }
{ "tool": "validate_service" }
{ "tool": "format_html" }
{ "tool": "finish", "answer": "your answer" }
For HTML requests, extract, validate, then format. Use tool results; do not invent data.
If a record is missing or invalid, explain that. For other questions, answer normally.
"""
question = "Show the Orders service as HTML."
print("User question:", question)

import json
data = {}
validated = False
history = ""

## Step 5 · Let the agent choose and use tools
Each turn sends the same system prompt, the question, and previous tool results. Python accepts only the listed tools and checks their order. The six-turn limit prevents an endless loop. The HTML is shown as text so you can inspect the tags. This example parses model-written JSON rather than using a provider-specific tool-calling API. Rerun Step 4 before each new question to reset the agent’s history.

In [ ]:
for turn in range(6):
    reply = client.ask(f"Question: {question}\nPrevious results:\n{history}", system=system_prompt)
    print("\nAgent:", reply)
    try:
        action = json.loads(reply)
    except json.JSONDecodeError:
        print("The model did not return JSON. Rerun the cell.")
        break
    if not isinstance(action, dict):
        print("Expected a JSON object. Rerun the cell.")
        break
    tool = action.get("tool")
    if tool == "extract_service" and isinstance(action.get("name"), str):
        data = extract_service(action["name"])
        validated = False
        result = data or "Service not found"
    elif tool == "validate_service":
        result = validate_service(data)
        validated = result == "Valid"
    elif tool == "format_html" and validated:
        result = format_html(data)
    elif tool == "finish" and isinstance(action.get("answer"), str):
        print("Final answer:", action["answer"])
        break
    else:
        result = "Unknown tool or invalid request; extract and validate before formatting."
    print("Tool result:", result)
    history += f"{reply}\nResult: {result}\n"
else:
    print("Stopped after six turns. Inspect the trace and try again.")

## What you should see
For the HTML request, look for three tool results: the Orders record, `Valid`, and HTML with its owner and responsibility. The model then gives an answer based on those results. Model choices can vary; read the trace.

## Try it
1. Change the question to ask for Inventory as HTML. Run Steps 4 and 5.
2. Ask for an unknown service. What does the extraction tool return?
3. Remove the Orders owner in Step 1 and rerun Steps 1–5. What does validation report?
4. Ask a conceptual question such as “What does a microservice own?” Does the model need a tool?

## Takeaway
A system prompt describes available tools. The model requests a tool, while application code controls execution and checks results.